# Task 1: Exploratory Data Analysis (EDA) on Titanic
**Tools:** Python, Pandas, NumPy  
**Goal:** Clean the Titanic dataset and extract key statistical insights.

## 1. Load the dataset

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

## 2. DataFrame operations: head, info, describe

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

## 3. Handling missing values

In [ ]:
missing = df.isnull().sum()
pd.DataFrame({'missing': missing, 'percent': (missing / len(df) * 100).round(2)}).sort_values('percent', ascending=False)

In [ ]:
# Age: fill with median
df['Age'] = df['Age'].fillna(df['Age'].median())

# Embarked: fill with mode (most frequent port)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin: ~77% missing, so drop the column
df = df.drop(columns=['Cabin'])

print(df.isnull().sum())

**Reasoning:** Age is numeric, so the median is robust to outliers. Embarked has only 2 missing values, so the mode is safe. Cabin is mostly empty, so it is dropped.

## 4. Handling duplicates

In [ ]:
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())
print("Final shape:", df.shape)

## 5. Feature engineering

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                        labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])
df[['Age', 'AgeGroup', 'FamilySize']].head()

## 6. Grouping and aggregation analysis

In [ ]:
print("Overall survival rate: {:.2%}".format(df['Survived'].mean()))
df.groupby('Sex')['Survived'].mean().round(3)

In [ ]:
df.groupby('Pclass')['Survived'].agg(['count', 'sum', 'mean']).round(3)

In [ ]:
df.groupby('AgeGroup', observed=True)['Survived'].mean().round(3)

In [ ]:
df.groupby('Embarked')['Survived'].mean().round(3)

In [ ]:
df.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack().round(3)

In [ ]:
df.groupby('Pclass').agg(avg_fare=('Fare', 'mean'),
                         avg_age=('Age', 'mean'),
                         avg_family=('FamilySize', 'mean')).round(2)

In [ ]:
df.groupby('FamilySize')['Survived'].agg(['count', 'mean']).round(3)

## 7. Key Insights
1. Overall, only about 38% of passengers survived.
2. Women survived at a far higher rate than men (about 74% vs 19%).
3. Survival dropped with passenger class: 1st class > 2nd > 3rd.
4. Children had higher survival rates than other age groups.
5. 1st-class passengers paid much higher fares and were older on average.
6. Medium family sizes (2-4) had better survival than solo travellers or very large families.

*(Verify these against your own outputs after running the notebook.)*

## 8. Save cleaned data

In [ ]:
df.to_csv('titanic_cleaned.csv', index=False)

## Conclusion
This task used Pandas to inspect, clean (missing values, duplicates), and aggregate the Titanic dataset. Sex, passenger class, and age were the strongest factors tied to survival.